In [ ]:
import open3d as o3d
import numpy as np
import fpsample
from collections import defaultdict

def generate_initial_mesh(pcd, sample_count=5000):
    """Generate initial mesh with FPS sampling and ball pivoting"""
    vertices = np.asarray(pcd.points)
    print("Original vertices shape:", vertices.shape)
    
    sampled_indices = fpsample.bucket_fps_kdtree_sampling(vertices, sample_count)
    sampled_pcd = pcd.select_by_index(sampled_indices)
    print(f"Sampled {sample_count} points")
    
    distances = np.linalg.norm(vertices[sampled_indices], axis=1)
    normalized_distances = (distances - np.min(distances)) / (np.max(distances) - np.min(distances))
    colors = 1 - np.repeat(normalized_distances[:, np.newaxis], 3, axis=1)
    sampled_pcd.colors = o3d.utility.Vector3dVector(colors)
    
    sampled_pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.05, max_nn=30))
    sampled_pcd.orient_normals_consistent_tangent_plane(k=30)
    

    distances = sampled_pcd.compute_nearest_neighbor_distance()
    avg_dist = np.mean(distances)
    radii = o3d.utility.DoubleVector([avg_dist, avg_dist*1.5, avg_dist*3])
    mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(sampled_pcd, radii)
    
    mesh = mesh.filter_smooth_taubin(number_of_iterations=30)
    mesh.compute_triangle_normals()
    mesh.compute_vertex_normals()
    
    return mesh

def get_triangle_adjacency(mesh):
    """Build adjacency dictionary mapping edges to triangles"""
    triangles = np.asarray(mesh.triangles)
    edge_to_triangles = defaultdict(list)
    
    for i, triangle in enumerate(triangles):
        edges = [
            tuple(sorted((triangle[0], triangle[1]))),
            tuple(sorted((triangle[1], triangle[2]))),
            tuple(sorted((triangle[2], triangle[0])))
        ]
        
        for edge in edges:
            edge_to_triangles[edge].append(i)
    
    return edge_to_triangles

def find_isolated_edges(mesh, edge_to_triangles):
    """Find edges that belong to only one triangle (isolated edges)"""
    isolated_edges = []
    for edge, tris in edge_to_triangles.items():
        if len(tris) == 1:
            isolated_edges.append((edge, tris[0]))
    return isolated_edges

def expand_mesh_targeted(mesh, original_pcd, max_iterations=5):
    """Célzott mesh bővítés, ami csak a szükséges kapcsolatokat hozza létre"""
    original_vertices = np.asarray(original_pcd.points)
    mesh_vertices = np.asarray(mesh.vertices)
    
    original_kdtree = o3d.geometry.KDTreeFlann(original_pcd)
    mesh_kdtree = o3d.geometry.KDTreeFlann(o3d.geometry.PointCloud())
    mesh_kdtree.set_geometry(o3d.geometry.PointCloud(o3d.utility.Vector3dVector(mesh_vertices)))
    
    for iteration in range(max_iterations):
        print(f"\n=== Iteráció {iteration+1} ===")
        
        edge_to_triangles = defaultdict(list)
        vertex_degree = defaultdict(int)
        triangles = np.asarray(mesh.triangles)
        
        for tri_idx, tri in enumerate(triangles):
            edges = [tuple(sorted((tri[0], tri[1]))),
                    tuple(sorted((tri[1], tri[2]))),
                    tuple(sorted((tri[2], tri[0])))]
            for edge in edges:
                edge_to_triangles[edge].append(tri_idx)
                vertex_degree[edge[0]] += 1
                vertex_degree[edge[1]] += 1
        
        new_triangles = []
        new_points = []
        processed_edges = set()
        
        for tri_idx, tri in enumerate(triangles):
            edges = [(tri[0], tri[1], tri[2]),
                    (tri[1], tri[2], tri[0]),
                    (tri[2], tri[0], tri[1])]
            
            for v1, v2, opposite_vertex in edges:
                edge = tuple(sorted((v1, v2)))
                
                if edge in processed_edges:
                    continue
                
                processed_edges.add(edge)

                if len(edge_to_triangles[edge]) > 1:
                    continue

                if vertex_degree.get(opposite_vertex, 0) >= 20:
                    continue

                [_, idx, _] = original_kdtree.search_knn_vector_3d(mesh_vertices[opposite_vertex], 30)
                
                for i in idx:
                    new_pos = original_vertices[i]
                    
                    [_, _, dists] = mesh_kdtree.search_knn_vector_3d(new_pos, 1)
                    if dists[0] < 1e-6:
                        continue

                    new_vertex_idx = len(mesh_vertices) + len(new_points)
                    new_points.append(new_pos)

                    new_triangles.append([opposite_vertex, new_vertex_idx, v1])
                    new_triangles.append([opposite_vertex, new_vertex_idx, v2])
                    
                    vertex_degree[opposite_vertex] += 2
                    vertex_degree[v1] += 1
                    vertex_degree[v2] += 1
                    vertex_degree[new_vertex_idx] = 2
                    
                    break
        
        if not new_points:
            print("Nincs több szabad oldal vagy megfelelő csúcs - kész!")
            break
            
        mesh.vertices.extend(new_points)
        new_tri_arr = np.vstack([triangles, np.array(new_triangles)])
        mesh.triangles = o3d.utility.Vector3iVector(new_tri_arr)

        mesh_vertices = np.asarray(mesh.vertices)
        mesh_kdtree.set_geometry(o3d.geometry.PointCloud(o3d.utility.Vector3dVector(mesh_vertices)))
        
        print(f"Hozzáadva: {len(new_points)} új pont és {len(new_triangles)} új háromszög")
        print(f"Összes háromszög: {len(new_tri_arr)}")

    mesh.compute_triangle_normals()
    mesh.compute_vertex_normals()
    mesh = mesh.filter_smooth_taubin(number_of_iterations=5)
    
    return mesh

def main():

    input_file = "centered.ply"
    pcd = o3d.io.read_point_cloud(input_file)
    
    initial_mesh = generate_initial_mesh(pcd, 5000)
    o3d.io.write_triangle_mesh("initial_mesh.ply", initial_mesh)
    
    expanded_mesh = expand_mesh_targeted(initial_mesh, pcd)

    o3d.io.write_triangle_mesh("expanded_mesh.ply", expanded_mesh)
    
    o3d.visualization.draw_geometries(
        [expanded_mesh],
        mesh_show_back_face=True,
        mesh_show_wireframe=False,
        window_name="Optimalizált 3D Modell"
    )

if __name__ == "__main__":
    main()


Original vertices shape: (395700, 3)
Sampled 5000 points
[Open3D WARNING] [KDTreeFlann::SetRawData] Failed due to no data.

=== Iteráció 1 ===
Hozzáadva: 2898 új pont és 5796 új háromszög
Összes háromszög: 10601

=== Iteráció 2 ===
Hozzáadva: 2234 új pont és 4468 új háromszög
Összes háromszög: 15069

=== Iteráció 3 ===
Nincs több szabad oldal vagy megfelelő csúcs - kész!
